# SIDD Medium 原始数据完整性检测

按顺序运行单元格。新会话无需先运行其他 Notebook。

### 1. 初始化

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
if (cwd / "src").is_dir():
    PROJECT_ROOT = cwd
elif (cwd.parent / "src").is_dir():
    PROJECT_ROOT = cwd.parent
else:
    raise RuntimeError("请从项目根目录或 notebooks/ 目录启动 Jupyter")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("项目目录：", PROJECT_ROOT)
sidd_root = PROJECT_ROOT / "data" / "SIDD_Medium_Srgb" / "Data"
if not sidd_root.is_dir():
    raise FileNotFoundError(sidd_root)


### 2. 场景与文件计数

In [ ]:
scene_dirs = sorted(p for p in sidd_root.iterdir() if p.is_dir())
noisy_files = sorted(sidd_root.rglob("*_NOISY_SRGB_*.PNG"))
gt_files = sorted(sidd_root.rglob("*_GT_SRGB_*.PNG"))
print("场景实例：", len(scene_dirs), "（期望 160）")
print("NOISY：", len(noisy_files), "（期望 320）")
print("GT：", len(gt_files), "（期望 320）")
scene_errors = []
for folder in scene_dirs:
    n = len(list(folder.glob("*_NOISY_SRGB_*.PNG")))
    g = len(list(folder.glob("*_GT_SRGB_*.PNG")))
    if (n, g) != (2, 2):
        scene_errors.append((folder.name, n, g))
print("场景内文件数量异常：", len(scene_errors))
if scene_errors:
    print(scene_errors[:10])


### 3. NOISY/GT 配对检查

In [ ]:
# 完整解码几百张超大 PNG 会耗时较长；按需修改开关。
FULL_DECODE = False
pair_errors = []
valid_pairs = 0
for noisy_path in noisy_files:
    gt_path = noisy_path.with_name(noisy_path.name.replace("_NOISY_SRGB_", "_GT_SRGB_"))
    if not gt_path.is_file():
        pair_errors.append((str(noisy_path), "GT 缺失"))
        continue
    try:
        with Image.open(noisy_path) as n, Image.open(gt_path) as g:
            if n.mode != "RGB" or g.mode != "RGB" or n.size != g.size:
                pair_errors.append((str(noisy_path), "尺寸或 RGB 模式异常"))
                continue
            if FULL_DECODE:
                n.load()
                g.load()
        valid_pairs += 1
    except (OSError, ValueError) as error:
        pair_errors.append((str(noisy_path), repr(error)))
print("配对检查完成：", valid_pairs, "对")
print("异常数量：", len(pair_errors))
if pair_errors:
    print(pair_errors[:10])
print("基础检查通过：", len(scene_dirs)==160 and len(noisy_files)==320 and len(gt_files)==320 and valid_pairs==320 and not scene_errors and not pair_errors)


### 4. 随机样本可视化（中心裁剪）

In [ ]:
random_noisy = random.choice(noisy_files)
random_gt = random_noisy.with_name(random_noisy.name.replace("_NOISY_SRGB_", "_GT_SRGB_"))
with Image.open(random_noisy) as n_img, Image.open(random_gt) as g_img:
    if n_img.size != g_img.size:
        raise ValueError("当前图像尺寸不一致")
    width, height = n_img.size
    crop = (width//2 - 128, height//2 - 128, width//2 + 128, height//2 + 128)
    noisy = np.asarray(n_img.crop(crop))
    gt = np.asarray(g_img.crop(crop))
score_p = peak_signal_noise_ratio(gt, noisy, data_range=255)
score_s = structural_similarity(gt, noisy, data_range=255, channel_axis=2)
print("样本：", random_noisy.parent.name, random_noisy.name)
print(f"中心裁剪原图 PSNR：{score_p:.2f} dB, SSIM：{score_s:.4f}")
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
for target, img, title in zip(ax, (noisy, gt), ("NOISY", "GT")):
    target.imshow(img); target.set_title(title); target.axis("off")
plt.show()


### 5. 可选：检查现有 DIV2K 目录的数量

In [ ]:
for folder_name, expected in (("DIV2K_train_HR", 800), ("DIV2K_valid_HR", 100)):
    folder = PROJECT_ROOT / "data" / folder_name
    if folder.is_dir():
        images = list(folder.glob("*.png"))
        print(folder_name, len(images), f"/ {expected}")
    else:
        print(folder_name, "未提供，跳过")
